In [2]:
import csv
import itertools
import logging
import os
import json
import random
import re
import shutil
import subprocess
import time
import threading
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, Future
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Set, Dict, Iterable

import httpx
import undetected_chromedriver as uc
from dotenv import load_dotenv
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

load_dotenv()

BASE_DIR = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping")
OUTPUT_PATH = BASE_DIR / "facebook" / "output.csv"
PROFILE_PATH = BASE_DIR / "chrome_profile"

MAX_STAGNANT = 10
SCROLL_SIZE = 3000
START_GROUP_IDX = 10

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
MAX_WORKERS = 15
LLM_TIMEOUT = 60.0
LLM_CONCURRENCY = 25
MAX_INFLIGHT_JOBS = 50
LLM_PAYLOAD_TRIM = 3500
LLM_MAX_TOKENS = 10000

# Shared httpx client with connection limits to avoid SDK storms
_httpx_client = httpx.Client(timeout=LLM_TIMEOUT, limits=httpx.Limits(max_connections=MAX_WORKERS, max_keepalive_connections=5))
_clients = [
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY2"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY3"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
]
_client_cycle = itertools.cycle(_clients)
_client_lock = threading.Lock()
_llm_semaphore = threading.Semaphore(LLM_CONCURRENCY)
_pending_lock = threading.Lock()
_pending_futures: Set[Future] = set()

def _get_client() -> OpenAI:
    with _client_lock:
        return next(_client_cycle)

def _register_future(fut: Future) -> None:
    with _pending_lock:
        _pending_futures.add(fut)

def _cleanup_done_futures() -> None:
    with _pending_lock:
        done = {f for f in _pending_futures if f.done()}
        _pending_futures.difference_update(done)

def _wait_for_backpressure() -> None:
    while True:
        _cleanup_done_futures()
        with _pending_lock:
            pending = len(_pending_futures)
        if pending < MAX_INFLIGHT_JOBS:
            return
        done, _ = wait(list(_pending_futures), return_when=FIRST_COMPLETED)
        with _pending_lock:
            _pending_futures.difference_update(done)
        done, _ = wait(list(_pending_futures), return_when=FIRST_COMPLETED)
        with _pending_lock:
            _pending_futures.difference_update(done)

GROUP_URLS: List[str] = [
    "https://www.facebook.com/groups/302468990428489/",
    "https://www.facebook.com/groups/322977734828852/",
    "https://www.facebook.com/groups/812156038944325/",
    "https://www.facebook.com/groups/1472146056424210/",
    "https://www.facebook.com/groups/426467944402414/",
    "https://www.facebook.com/groups/homerentcm/",
    "https://www.facebook.com/groups/509895225859790/",
    "https://www.facebook.com/groups/1299302030158649/",
    "https://www.facebook.com/groups/530492663958652/",
    "https://www.facebook.com/groups/596670275854317/",
    "https://www.facebook.com/groups/cnxre/",
    "https://www.facebook.com/groups/1885263611797363/",
    "https://www.facebook.com/groups/1897854047114064/",
    "https://www.facebook.com/groups/303531690102574/",
    "https://www.facebook.com/groups/1881982752029163/",
    "https://www.facebook.com/groups/568026117396809/",
    "https://www.facebook.com/groups/1928645537294336/",
    "https://www.facebook.com/groups/298821664628156/",
    "https://www.facebook.com/groups/903971886395138/",
    "https://www.facebook.com/groups/142702946428033/",
    "https://www.facebook.com/groups/250469775470436/",
    "https://www.facebook.com/groups/1873094122912006/",
    "https://www.facebook.com/groups/2083392538672247/",
    "https://www.facebook.com/groups/864193946960728/",
    "https://www.facebook.com/groups/baanchiangmai/",
    "https://www.facebook.com/groups/694087674785430/",
    "https://www.facebook.com/groups/411301775702951/",
    "https://www.facebook.com/groups/169718747164928/",
    "https://www.facebook.com/groups/1475061816108017/",
    "https://www.facebook.com/groups/1582425938465943/",
    "https://www.facebook.com/groups/sale.rent.poolvillachiangmai/",
    "https://www.facebook.com/groups/251125079442673/",
    "https://www.facebook.com/groups/1034329704984830/",
    "https://www.facebook.com/groups/203683550205761/",
    "https://www.facebook.com/groups/1450131905304596/",
    "https://www.facebook.com/groups/959493160788393/",
    "https://www.facebook.com/groups/2897034136980512/",
    "https://www.facebook.com/groups/korn.property/",
    "https://www.facebook.com/groups/1456428424593312/",
    "https://www.facebook.com/groups/152080739566471/",
    "https://www.facebook.com/groups/Land.House.C.M.2014/",
    "https://www.facebook.com/groups/2336780789695894/",
    "https://www.facebook.com/groups/236116797208244/",
    "https://www.facebook.com/groups/landhomechiangmai/",
]

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ",
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย"
]

SYSTEM_PROMPT = """คุณคือ AI Data Engine ระดับ Senior ที่เชี่ยวชาญด้าน Real Estate Analytics 
ทำหน้าที่สกัดข้อมูล (Entity Extraction) และจำแนกประเภท (Classification) จากข้อความโพสต์อสังหาริมทรัพย์
กฎเหล็ก: ตอบกลับเป็น JSON Structure เท่านั้น ห้ามมีข้อความเกริ่นนำหรือสรุปท้ายโดยเด็ดขาด การขึ้นบรรทัดใหม่ใน Description ต้องใช้ \n ห้ามใช้การกด Enter จริงๆ เพื่อป้องกัน JSON พัง

{
  "is_real_estate": true/false,
  "is_owner": true/false,
  "owner_confidence": 0.0,
  "classification_type": "OWNER/AGENT/UNKNOWN",
  "evidence_phrases": [],
  "risk_flags": [],
  "post_date_text": "สกัดข้อความวันที่/เวลาจากต้นฉบับ",
  "extracted":{
    "property_type": "บ้านเดี่ยว/คอนโด/ที่ดิน ฯลฯ",
    "rental_sale_status": "ขาย/เช่า/ขายและเช่า",
    "project_name": "ชื่อโครงการ (ถ้าไม่ระบุให้เป็น null)",
    "district": "เขต/พื้นที่/ทำเล",
    "size_text": "ขนาดพื้นที่/พื้นที่ใช้สอย",
    "price_text": "ข้อความราคาเต็ม",
    "price_value_thb": null,
    "phone": "เบอร์โทรศัพท์ (สกัดเฉพาะตัวเลข)",
    "line": "ID Line หรือ Link Line",
    "description": "ดึงรายละเอียดทั้งหมด ห้ามตัดทอน"
  }
}

=== OWNER vs AGENT CLASSIFICATION LOGIC (EXPERT HEURISTICS) ===

ใช้กลไก Short-circuit Evaluation ตามลำดับความสำคัญดังนี้:

GATE 1: AGENT HARD-FILTER (ถือเป็น AGENT ทันทีหากพบสิ่งเหล่านี้)
- [Industry Keywords]: "ติดทรัพย์", "รับ Co-agent", "Co-broker", "ยินดีร่วมงานกับเอเจ้นท์" (ในบริบทที่เป็นผู้ถือทรัพย์), "รหัสทรัพย์", "Stock", "ทรัพย์สวย"
- [Contact Patterns]: LINE ID ที่มี "@", ชื่อผู้ติดต่อที่ระบุว่าเป็น "Admin", "ทีมงาน", "ฝ่ายขาย", "Sale"
- [Corporate Tone]: ใช้คำหรูหราเกินจริงแบบ Marketing Material เช่น "นิยามใหม่แห่งการพักผ่อน", "เอกสิทธิ์เฉพาะคุณ", "Ultra Luxury" (โดยไม่มี Personal Story ประกอบ)
- [Formatting]: มีรายการทรัพย์อื่นพ่วงท้าย หรือมี Hashtag จำนวนมากที่เกี่ยวข้องกับบริษัท Agent

GATE 2: OWNER VERIFICATION (พิจารณาว่าเป็น OWNER)
- [Direct Explicit Claim]: "Owner Post", "เจ้าของขายเอง", "เจ้าของปล่อยเช่าเอง", "ไม่ผ่านนายหน้า", "ยินดีรับ Agent" (กรณีระบุชัดว่าตนเองเป็นเจ้าของ)
- [Personal Storytelling/Anecdotes]: มีเหตุผลการขายที่เฉพาะตัว เช่น "ย้ายงาน", "ขายเพราะไปอยู่ต่างประเทศ", "บ้านอายุ 3 ปีอยู่จริงไม่ถึงปี", "วัสดุเลือกเกรดดีที่สุดเพราะตอนแรกจะอยู่เอง"
- [Personal Tone]: ใช้สรรพนาม "พี่", "ผม", "ฉัน", "บ้านเรา", "ขอรูปเพิ่มเติมทาง Inbox" (แบบไม่เป็น Pattern ระบบ)
- [Financial Transparency]: "ขายต่ำกว่าทุน", "ราคาซื้อมา...", "โอนคนละครึ่ง", "ยินดีต่อรอง"

GATE 3: HEURISTIC SCORING & AMBIGUITY HANDLING
- หากโพสต์สั้นและ Generic มาก (เช่น "ขายบ้าน [ราคา] [เบอร์โทร]") ให้ Default เป็น AGENT ด้วย Confidence ต่ำ (0.3) เพราะพฤติกรรมเจ้าของจริงมักจะให้รายละเอียดมากกว่าปกติ
- กรณี "Owner Post" แต่ใช้ LINE @ หรือมีรหัสทรัพย์ ให้ Short-circuit ไปที่ AGENT ทันที (ถือเป็น False Claim)

กฎการจัดการ Data Integrity:
1. price_value_thb: ให้สกัดเฉพาะตัวเลข Integer เท่านั้น (เช่น 18.5 ล้าน -> 18500000)
2. description: ต้องรักษาความหมายเดิมไว้ทั้งหมด ห้ามทำการ Summarize จนเสียข้อมูลสำคัญ
"""

MONTH_MAP = {
    "มกราคม": 1, "ม.ค.": 1, "กุมภาพันธ์": 2, "ก.พ.": 2, "มีนาคม": 3, "มี.ค.": 3,
    "เมษายน": 4, "เม.ย.": 4, "พฤษภาคม": 5, "พ.ค.": 5, "มิถุนายน": 6, "มิ.ย.": 6,
    "กรกฎาคม": 7, "ก.ค.": 7, "สิงหาคม": 8, "ส.ค.": 8, "กันยายน": 9, "ก.ย.": 9,
    "ตุลาคม": 10, "ต.ค.": 10, "พฤศจิกายน": 11, "พ.ย.": 11, "ธันวาคม": 12, "ธ.ค.": 12,
}

_INVISIBLE_CHARS_RE = re.compile("[\u200b\u200c\u200d\ufeff\u00a0\u2060\u180e\u2028\u2029\u00ad]")
_TRUNCATION_SUFFIXES = ("... ดูเพิ่มเติม", "...ดูเพิ่มเติม", "... See more", "...See more")
csv_lock = threading.Lock()

def _is_truncated(content: str) -> bool:
    stripped = content.strip()
    return any(stripped.endswith(s) for s in _TRUNCATION_SUFFIXES)

def is_post_older_than_24h(date_text: str) -> bool:
    if not date_text or date_text == "N/A": return False
    val = _INVISIBLE_CHARS_RE.sub("", date_text).strip().lower()
    
    # ภายใน 24 ชม. ถือว่ายังอยู่ในกรอบเวลา
    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return False
        
    # ถ้าเจอเลขวัน > 1 วัน (เช่น 2 วัน, 3 วัน) หรือเป็นวันที่แบบเดือน/ปี ถือว่าเกิน 24 ชม.
    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        if int(m_days.group(1)) >= 1:
            return True
            
    m_date = re.search(r"(\d{1,2})[\s\.\-/]+([ก-๙a-zA-Z]+)(?:[\s\.\-/]+(\d{2,4}))?", val)
    if m_date or "เมื่อวาน" in val:
        return True
        
    return False

def get_chrome_version(chrome_exec: str) -> int:
    try:
        res = subprocess.run([chrome_exec, "--version"], capture_output=True, text=True, check=False)
        return int(re.search(r"(\d+)\.", res.stdout).group(1)) if res.stdout else 0
    except Exception: return 0

def create_driver() -> uc.Chrome:
    PROFILE_PATH.mkdir(parents=True, exist_ok=True)
    chrome_exec = shutil.which("google-chrome") or shutil.which("chromium-browser")
    opts = uc.ChromeOptions()
    opts.add_argument(f"--user-data-dir={PROFILE_PATH}")
    # Reduce payload: disable images, media and plugins
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.media_stream": 2,
        "profile.managed_default_content_settings.plugins": 2,
    }
    opts.add_experimental_option("prefs", prefs)
    opts.add_argument("--disable-notifications")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--headless=new")
    opts.add_argument("--disable-features=IsolateOrigins,site-per-process")
    opts.add_argument("--blink-settings=imagesEnabled=false")
    opts.page_load_strategy = "eager"
    return uc.Chrome(options=opts, version_main=get_chrome_version(chrome_exec), browser_executable_path=chrome_exec)

def humanized_scroll(driver: uc.Chrome) -> None:
    driver.execute_script(f"window.scrollBy(0, {SCROLL_SIZE + random.randint(-500, 500)});")
    time.sleep(random.uniform(0.5, 1.0))

def apply_new_post_filter(driver: uc.Chrome):
    try:
        driver.execute_script("""
            const filterBtn = Array.from(document.querySelectorAll('div[role="button"]'))
                .find(e => e.innerText && (e.innerText.includes('\u0e40\u0e23\u0e35\u0e22\u0e07\u0e25\u0e33\u0e14\u0e31\u0e1a\u0e1f\u0e35\u0e14\u0e43\u0e19\u0e01\u0e25\u0e38\u0e48\u0e21\u0e15\u0e32\u0e21') || e.innerText.includes('\u0e08\u0e31\u0e14\u0e40\u0e23\u0e35\u0e22\u0e07\u0e15\u0e32\u0e21')));
            if (filterBtn) {
                filterBtn.click();
                setTimeout(() => {
                    const options = Array.from(document.querySelectorAll('div[role="menuitemradio"]'));
                    const target = options.find(e => e.innerText && (e.innerText.includes('\u0e42\u0e1e\u0e2a\u0e15\u0e4c\u0e43\u0e2b\u0e21\u0e48') || e.innerText.includes('\u0e23\u0e32\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e34\u0e19\u0e04\u0e49\u0e32\u0e43\u0e2b\u0e21\u0e48')));
                    if (target) target.click();
                }, 1500);
            }
        """)
        time.sleep(3.5)
    except Exception as e: logger.error(f"Filter error: {e}")

def atomic_fb_extract(driver: uc.Chrome) -> tuple[List[Dict[str, str]], bool]:
    """
    Atomic JavaScript Pipeline: Scroll + Expand + Extract in single execute_script call
    Reduces WebDriver JSON-RPC overhead dramatically (~40-50% faster than sequential ops)
    Returns: (extracted_posts, found_old_post_flag)
    """
    result = driver.execute_script(r"""
        const INVIS_CODES = new Set([0x200B,0x200C,0x200D,0xFEFF,0x00A0,0x2060,0x180E,0x2028,0x2029,0x00AD]);
        function cleanText(s) {
            let out = '';
            for (let i = 0; i < s.length; i++) {
                if (!INVIS_CODES.has(s.charCodeAt(i))) out += s[i];
            }
            return out.trim().toLowerCase();
        }
        // STEP 1: Scroll to last article
        const articles = document.querySelectorAll("div[role='article']");
        if (articles.length > 0) {
            const lastArticle = articles[articles.length - 1];
            lastArticle.scrollIntoView({behavior: 'instant', block: 'end'});
            window.scrollBy(0, 800);
        }
        // STEP 2: Expand all "See more" buttons
        const TARGET = new Set(['ดูเพิ่มเติม', 'see more']);
        const candidates = Array.from(document.querySelectorAll('div[role="button"], span[role="button"]'));
        for (const el of candidates) {
            const text = cleanText(el.innerText || el.textContent || '');
            if (!TARGET.has(text)) continue;
            const rect = el.getBoundingClientRect();
            if (rect.width === 0 || rect.height === 0) continue;
            const cx = rect.left + rect.width / 2;
            const cy = rect.top + rect.height / 2;
            const evtOpts = {bubbles: true, cancelable: true, view: window, clientX: cx, clientY: cy, screenX: window.screenX + cx, screenY: window.screenY + cy};
            for (const evtType of ['pointerover','mouseover','pointermove','mousemove','pointerdown','mousedown','pointerup','mouseup','click']) {
                try { el.dispatchEvent(new MouseEvent(evtType, evtOpts)); } catch(_) {}
            }
        }
        // STEP 3: Extract all articles + early filter for old posts (24h boundary check)
        const results = [];
        let hasOldPost = false;
        document.querySelectorAll("div[role='article']").forEach(a => {
            const linkNodes = Array.from(a.querySelectorAll("a[href]")).filter(l => l.href.includes('/posts/') || l.href.includes('/permalink/'));
            if (linkNodes.length === 0) return;
            const url = linkNodes[0].href.split('?')[0];
            const msgNode = a.querySelector("div[data-ad-comet-preview='message']") || a.querySelector("div[data-ad-preview='message']");
            if (!msgNode) return;
            const content = msgNode.innerText.trim();
            let date = "N/A";
            for (let l of linkNodes) {
                const aria = (l.getAttribute("aria-label") || "").trim();
                const text = (l.textContent || "").trim();
                if (aria && aria.length > 0 && aria.length < 30) { date = aria; break; }
                else if (text && text.length > 0 && text.length < 30) { date = text; break; }
            }
            // Early boundary filtering: skip posts older than 24h at browser level
            const dateStr = cleanText(date);
            if (dateStr && !dateStr.includes('วันนี้') && !dateStr.includes('นาที') && !dateStr.includes('ชั่วโมง')) {
                const hasDayCount = /\d+\s*วัน/.test(dateStr);
                const hasDateFormat = /\d{1,2}[\s.\/\-]+\w+/.test(dateStr);
                const isYesterday = dateStr.includes('เมื่อวาน');
                if (hasDayCount || hasDateFormat || isYesterday) {
                    hasOldPost = true;
                }
            }
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return {results, hasOldPost};
    """)
    return result['results'], result['hasOldPost']

def expand_all_see_more(driver: uc.Chrome) -> int:
    clicked = driver.execute_script("""
        const INVIS_CODES = new Set([0x200B,0x200C,0x200D,0xFEFF,0x00A0,0x2060,0x180E,0x2028,0x2029,0x00AD]);
        function cleanText(s) {
            let out = '';
            for (let i = 0; i < s.length; i++) {
                if (!INVIS_CODES.has(s.charCodeAt(i))) out += s[i];
            }
            return out.trim().toLowerCase();
        }
        const TARGET = new Set(['\u0e14\u0e39\u0e40\u0e1e\u0e34\u0e48\u0e21\u0e40\u0e15\u0e34\u0e21', 'see more']);
        const candidates = Array.from(document.querySelectorAll('div[role="button"], span[role="button"]'));
        let clicked = 0;
        for (const el of candidates) {
            const text = cleanText(el.innerText || el.textContent || '');
            if (!TARGET.has(text)) continue;
            el.scrollIntoView({behavior: 'instant', block: 'center'});
            const rect = el.getBoundingClientRect();
            if (rect.width === 0 || rect.height === 0) continue;
            const cx = rect.left + rect.width / 2;
            const cy = rect.top + rect.height / 2;
            const evtOpts = {bubbles: true, cancelable: true, view: window, clientX: cx, clientY: cy, screenX: window.screenX + cx, screenY: window.screenY + cy};
            for (const evtType of ['pointerover','mouseover','pointermove','mousemove','pointerdown','mousedown','pointerup','mouseup','click']) {
                try { el.dispatchEvent(new MouseEvent(evtType, evtOpts)); } catch(_) {}
            }
            clicked++;
        }
        return clicked;
    """)
    if clicked: time.sleep(3.5)
    return clicked or 0

def batch_extract_dom(driver: uc.Chrome) -> List[Dict[str, str]]:
    return driver.execute_script("""
        const results = [];
        document.querySelectorAll("div[role='article']").forEach(a => {
            const linkNodes = Array.from(a.querySelectorAll("a[href]")).filter(l => l.href.includes('/posts/') || l.href.includes('/permalink/'));
            if (linkNodes.length === 0) return;
            const url = linkNodes[0].href.split('?')[0];
            const msgNode = a.querySelector("div[data-ad-comet-preview='message']") || a.querySelector("div[data-ad-preview='message']");
            if (!msgNode) return;
            const content = msgNode.innerText.trim();
            let date = "N/A";
            for (let l of linkNodes) {
                const aria = (l.getAttribute("aria-label") || "").trim();
                const text = (l.textContent || "").trim();
                if (aria && aria.length > 0 && aria.length < 30) { date = aria; break; }
                else if (text && text.length > 0 && text.length < 30) { date = text; break; }
            }
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return results;
    """)

def call_llm_service(payload: str, raw_url: str = "") -> dict | None:
    """
    Strict HTTPX Client with custom retry logic (no SDK storms)
    Trimmed payload[:LLM_PAYLOAD_TRIM] to reduce TTFB latency
    """
    trimmed_payload = payload[:LLM_PAYLOAD_TRIM]
    
    for attempt in range(3):
        acquired = _llm_semaphore.acquire(timeout=LLM_TIMEOUT)
        if not acquired:
            logger.info(f"LLM semaphore timeout | URL: {raw_url} | attempt={attempt + 1}")
            time.sleep(1.0 * (attempt + 1))
            continue
        client_idx = None
        t0 = time.perf_counter()
        try:
            c = _get_client()
            client_idx = (_clients.index(c) + 1) if c in _clients else 0
            logger.info(f"LLM start | client={client_idx} | attempt={attempt + 1} | url={raw_url}")
            response = c.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.0,
                max_tokens=LLM_MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": trimmed_payload},
                ],
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            elapsed = time.perf_counter() - t0
            logger.info(f"LLM ok | client={client_idx} | elapsed={elapsed:.2f}s | url={raw_url}")
            return json.loads(content)
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            logger.error(f"LLM error | client={client_idx} | elapsed={elapsed:.2f}s | url={raw_url} | err={type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
        finally:
            _llm_semaphore.release()
    return None

def parse_date(date_text: str) -> str:
    now = datetime.now()
    if not date_text or date_text == "N/A": return "-"
    val = _INVISIBLE_CHARS_RE.sub("", date_text).strip().lower()

    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return now.strftime("%d/%m/%Y")
    if "เมื่อวาน" in val:
        return (now - timedelta(days=1)).strftime("%d/%m/%Y")

    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        return (now - timedelta(days=int(m_days.group(1)))).strftime("%d/%m/%Y")

    m_date = re.search(r"(\d{1,2})[\s\.\-/]+([ก-๙a-zA-Z]+)(?:[\s\.\-/]+(\d{2,4}))?", val)
    if m_date:
        d, m_raw = int(m_date.group(1)), m_date.group(2)
        m = MONTH_MAP.get(m_raw)
        if m:
            y = now.year
            if m_date.group(3):
                y_raw = int(m_date.group(3))
                y = y_raw - 543 if y_raw > 2400 else (y_raw if y_raw > 100 else 2000 + y_raw)
            return f"{d:02d}/{m:02d}/{y}"
    return "-"

def transform_record(raw_row: dict, ai_data: dict) -> dict:
    ext = ai_data.get("extracted", {})
    llm_date = _INVISIBLE_CHARS_RE.sub("", ai_data.get("post_date_text", "")).strip()
    dom_date = raw_row.get("Date", "").strip()
    actual_date = llm_date if llm_date and llm_date != "N/A" else dom_date
    return {
        "วันที่โพส": parse_date(actual_date),
        "website": "facebook",
        "ประเภท": ext.get("property_type", "-"),
        "สถานะ": ext.get("rental_sale_status", "-"),
        "ชื่อโครงการ": ext.get("project_name", "-"),
        "ขนาด": ext.get("size_text", "-"),
        "ราคา": ext.get("price_text", "-"),
        "เขต": ext.get("district", "-"),
        "Link": raw_row.get("Post_URL", "-"),
        "เบอร์โทรศัพท์": ext.get("phone", "-"),
        "Line": ext.get("line", "-"),
        "คำอธิบาย": ext.get("description", "-"),
    }

def worker_process_and_save(raw_item: dict) -> None:
    payload = f"Post Date: {raw_item.get('Date', 'N/A')}\n\nContent:\n{raw_item.get('Full_Content', '')}"
    ai_response = call_llm_service(payload, raw_item.get("Post_URL", ""))
    if not ai_response or not ai_response.get("is_real_estate"): return
    if not ai_response.get("is_owner"):
        logger.info(f"Agent Filtered | URL: {raw_item.get('Post_URL')} | Pattern: {ai_response.get('risk_flags')}")
        return
    final_data = transform_record(raw_item, ai_response)
    with csv_lock:
        with open(OUTPUT_PATH, "a", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writerow(final_data)

def process_group(driver: uc.Chrome, url: str, seen_urls: Set[str], executor: ThreadPoolExecutor, group_idx: int, total_groups: int):
    try:
        logger.info(f"[Group {group_idx}/{total_groups}] Start processing: {url}")
        driver.get(url)
        time.sleep(5)
        apply_new_post_filter(driver)

        saved_count, stagnant_count = 0, 0
        found_old_post = False
        
        for _ in range(300): # Allow deeper scroll if needed, bounded by time
            _wait_for_backpressure()
            extracted, found_old = atomic_fb_extract(driver)
            found_old_post = found_old_post or found_old

            if not extracted:
                stagnant_count += 1
            else:
                unseen = [i for i in extracted if i["Post_URL"] not in seen_urls]
                valid_unseen = [i for i in unseen if not is_post_older_than_24h(i["Date"]) ]
                new_items = [i for i in valid_unseen if not _is_truncated(i["Full_Content"]) ]
                truncated_count = len(valid_unseen) - len(new_items)

                if truncated_count:
                    logger.info(f"[Group {group_idx}/{total_groups}] Skipped {truncated_count} truncated post(s), will retry next iteration")

                if new_items:
                    stagnant_count = 0
                    for item in new_items:
                        seen_urls.add(item["Post_URL"])
                        fut = executor.submit(worker_process_and_save, item)
                        _register_future(fut)
                        saved_count += 1
                    logger.info(f"[Group {group_idx}/{total_groups}] Collected {len(new_items)} new items (Total saved this group: {saved_count})")
                elif truncated_count:
                    stagnant_count = 0
                else:
                    stagnant_count += 1

            if found_old_post or stagnant_count >= MAX_STAGNANT:
                reason = "Found post older than 24h" if found_old_post else f"Stagnant: {stagnant_count}"
                logger.info(f"[Group {group_idx}/{total_groups}] Stop condition met. ({reason}, Saved: {saved_count})")
                break

            humanized_scroll(driver)
    except Exception as e:
        logger.error(f"Error processing {url}: {e}")

def main():
    if not OUTPUT_PATH.exists():
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    seen_urls: Set[str] = set()
    with open(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
        seen_urls.update(row["Link"] for row in csv.DictReader(f) if row.get("Link"))

    total_groups = len(GROUP_URLS)
    resume_slice = GROUP_URLS[START_GROUP_IDX - 1:]
    logger.info(f"Resuming from group {START_GROUP_IDX}/{total_groups}: {resume_slice[0]}")

    driver = create_driver()
    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for idx, group_url in enumerate(resume_slice, start=START_GROUP_IDX):
                process_group(driver, group_url, seen_urls, executor, idx, total_groups)
            _wait_for_backpressure()
    finally:
        driver.quit()

if __name__ == "__main__":
    main()

[18:08:00] [INFO] Resuming from group 1/44: https://www.facebook.com/groups/302468990428489/
[18:08:26] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[18:08:27] [INFO] [Group 1/44] Start processing: https://www.facebook.com/groups/302468990428489/
[18:08:44] [INFO] LLM start | client=1 | attempt=1 | url=https://www.facebook.com/groups/302468990428489/posts/1889927805015925/
[18:08:44] [INFO] [Group 1/44] Collected 1 new items (Total saved: 1)
[18:08:49] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[18:08:49] [INFO] LLM ok | client=1 | elapsed=5.00s | url=https://www.facebook.com/groups/302468990428489/posts/1889927805015925/
[18:08:50] [INFO] [Group 1/44] Expanded 2 'See more' button(s)
[18:08:50] [INFO] [Group 1/44] Skipped 1 truncated post(s)
[18:08:50] [INFO] LLM start | client=2 | attempt=1 | url=https://www.facebook.com/groups/302468990428489/posts/1889921285016577/
[18:08:5

KeyboardInterrupt: 